# Design exercise D3 · Defects on the tile line — Part B

The new camera photographs every tile, and the inspectors re-check the images
in order of your score, as many as each shift allows. Your job: give each of
2,000 new images a score, so that the defective tiles come as high in the
ranking as possible.

You have about 15 minutes. The cells below load the data, define the metric,
validate two baselines on the pilot images and write the file. Make **your**
design choices in the cell marked *Your model*, compare them on the pilot,
then hand in.

**Scoring.** Your scores are compared with the real outcomes, which only the
grader has, by average precision. It gives the points: 0 at the level of a
random ranking (0.07), 10 at the reference model's 0.75 or above, linear in
between.

> **One attempt.** Only the **first** file you turn in is scored. If you
> unsubmit and turn in another file, it is not scored: you keep the score of
> the first. Choose your model with the notebook's own validation, check the file with the last
> cell, and only then turn it in.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

BASE = "https://raw.githubusercontent.com/fabsilvestri/AIML-Course/main/design/d03/"
archive = pd.read_csv(BASE + "archive.csv")   # old station, labelled
pilot = pd.read_csv(BASE + "pilot.csv")       # new camera, labelled by hand
new = pd.read_csv(BASE + "test.csv")          # new camera: the 2,000 to score

PIXELS = [c for c in archive.columns if c.startswith("p")]
assert len(PIXELS) == 16 * 16
assert archive.shape == (4000, 258) and pilot.shape == (600, 258)
assert new.shape == (2000, 257)
for name, df in [("archive", archive), ("pilot", pilot)]:
    print(f"{name}: {len(df):,} images, {df['defect'].mean():.1%} defective")

Each row is one 16 by 16 greyscale image, `p00_00` to `p15_15` (row, then
column), brightness 0 to 255. Look at a few before anything else.

In [ ]:
import matplotlib.pyplot as plt

def show(df, title):
    fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
    for ax, (_, row) in zip(axes, df.head(8).iterrows()):
        ax.imshow(row[PIXELS].to_numpy(float).reshape(16, 16), cmap="gray",
                  vmin=0, vmax=255)
        ax.axis("off")
    fig.suptitle(title)
    plt.show()

show(archive[archive["defect"] == 1], "archive, defective")
show(pilot[pilot["defect"] == 1], "pilot, defective")
show(archive[archive["defect"] == 0], "archive, good")

The metric is the grader's: average precision of the ranking. Models are
compared on the **pilot** images, which come from the new camera like the
2,000 you will score. The pixels are standardised with one mean and one
standard deviation, computed on the archive (the training images) only.

In [ ]:
def average_precision(y, scores):
    """Mean, over the defective images, of the precision at each one's rank."""
    order = np.argsort(-np.asarray(scores), kind="stable")
    hit = np.asarray(y)[order]
    precision = np.cumsum(hit) / np.arange(1, len(hit) + 1)
    return precision[hit == 1].mean()

X_arch = archive[PIXELS].to_numpy(np.float32)
y_arch = archive["defect"].to_numpy()
X_pilot = pilot[PIXELS].to_numpy(np.float32)
y_pilot = pilot["defect"].to_numpy()
X_new = new[PIXELS].to_numpy(np.float32)

MEAN, SD = X_arch.mean(), X_arch.std()        # training images only

def as_images(X):
    """(n, 256) pixels -> (n, 1, 16, 16) standardised tensor."""
    return torch.tensor((X - MEAN) / SD).reshape(-1, 1, 16, 16)

print(f"random ranking on the pilot: "
      f"{average_precision(y_pilot, np.random.default_rng(0).random(len(y_pilot))):.3f}")

A first baseline: logistic regression on the 256 pixels, fitted on the
archive.

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
logreg.fit(X_arch, y_arch)
print(f"logistic regression, pilot AP: "
      f"{average_precision(y_pilot, logreg.predict_proba(X_pilot)[:, 1]):.3f}")

A training loop for any network that maps a batch `(n, 1, 16, 16)` to `(n, 1)`
scores: Adam, binary cross-entropy on the logits, batches of 64. `augment`, if
you give one, receives each training batch as a tensor `(n, 1, 16, 16)` and
must return one of the same shape. Scores are computed in `eval()` mode, with
no gradient.

The second baseline is a small convolutional network: four convolutions with
batch normalisation, three pooling stages, then a flatten and a dense head.

In [ ]:
def fit(make_net, X, y, epochs=15, augment=None, seed=0):
    torch.manual_seed(seed)
    net = make_net()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    images = as_images(X)
    target = torch.tensor(y, dtype=torch.float32)
    for epoch in range(epochs):
        net.train()
        for batch in torch.randperm(len(images)).split(64):
            xb = images[batch]
            if augment is not None:
                xb = augment(xb)
            opt.zero_grad()
            loss = loss_fn(net(xb).squeeze(1), target[batch])
            loss.backward()
            opt.step()
    return net

def scores(net, X):
    net.eval()
    with torch.no_grad():
        return net(as_images(X)).squeeze(1).numpy()

def pilot_ap(make_net, augment=None, seed=0):
    """Fit on the archive, score the pilot."""
    net = fit(make_net, X_arch, y_arch, augment=augment, seed=seed)
    return average_precision(y_pilot, scores(net, X_pilot))

def block(c_in, c_out):
    return [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU()]

def make_baseline():
    return nn.Sequential(
        *block(1, 16),
        *block(16, 32), nn.MaxPool2d(2),          # 32 x 8 x 8
        *block(32, 32), nn.MaxPool2d(2),          # 32 x 4 x 4
        *block(32, 32), nn.MaxPool2d(2),          # 32 x 2 x 2
        nn.Flatten(), nn.Linear(32 * 2 * 2, 32), nn.ReLU(), nn.Linear(32, 1))

assert make_baseline()(torch.zeros(2, 1, 16, 16)).shape == (2, 1)
print(f"baseline CNN, pilot AP: {pilot_ap(make_baseline):.3f}")

## Your model

Build your design here, and compare it with the baselines on the pilot. Every
choice — the architecture, its head, any augmentation, the number of epochs —
is yours, and is judged by `pilot_ap`. One run is one seed: before you trust a
small difference, try another `seed`.

In [ ]:
def make_net():
    return make_baseline()          # replace with your design

augment = None                      # or a function of a batch, see above

assert make_net()(torch.zeros(2, 1, 16, 16)).shape == (2, 1)
print(f"your model, pilot AP: {pilot_ap(make_net, augment):.3f}")

## Hand in

Fit the chosen design on all 4,600 labelled images (archive and pilot), score
the 2,000 new ones, and check the file before you write it.

In [ ]:
net = fit(make_net, np.vstack([X_arch, X_pilot]), np.r_[y_arch, y_pilot],
          augment=augment)
score = scores(net, X_new)

out = pd.DataFrame({"image_id": new["image_id"], "score": score})
assert len(out) == 2000 and out["image_id"].is_unique
assert out["score"].notna().all() and np.isfinite(out["score"]).all()
assert out["score"].nunique() > 1000, "too many ties to rank the images"
out.to_csv("predictions.csv", index=False)
print("wrote predictions.csv:", len(out), "rows")

In Colab, download it (the folder icon on the left, or run the next cell),
then in Classroom open the assignment, **Add → File**, choose
`predictions.csv`, and **Turn in**. The grader marks it within the hour.
**Only this first file counts**: a second one is not scored.

In [ ]:
try:
    from google.colab import files
    files.download("predictions.csv")
except ImportError:
    print("Not in Colab: predictions.csv is in the working directory.")